# Fall Army Worm Detection — Training Notebook

Train a binary classifier (**infected** vs **non_infected**) using MobileNetV2 transfer learning.

**How to use:**
1. Upload your `dataset/` folder to Google Drive (with subfolders: `Blight`, `Common_Rust`, `Gray_Leaf_Spot`, `Healthy`, `worm`)
2. Set `DRIVE_DATASET_PATH` in the config cell below
3. Go to **Runtime → Change runtime type → GPU (T4)**
4. **Run All** cells
5. Download the trained model from the last cell

## 1. Setup

In [ ]:
!pip install -q matplotlib seaborn scikit-learn

## 2. Configuration

In [ ]:
# ============================================================
# EDIT THIS: Path to your dataset folder on Google Drive
# The folder should contain subfolders: Blight, Common_Rust,
# Gray_Leaf_Spot, Healthy, worm
# ============================================================
DRIVE_DATASET_PATH = "/content/drive/MyDrive/FAW_dataset/dataset"

# Training hyperparameters
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS_FROZEN = 15       # Phase 1: frozen backbone
EPOCHS_FINETUNE = 15     # Phase 2: fine-tune top layers
LR_FROZEN = 1e-3
LR_FINETUNE = 1e-4
FINE_TUNE_LAYERS = 30    # Unfreeze top N layers of MobileNetV2
DROPOUT_RATE = 0.3
FOCAL_GAMMA = 2.0

# Binary class config
POSITIVE_CLASS = "worm"          # source folder name
POSITIVE_LABEL = "infected"      # target label
NEGATIVE_LABEL = "non_infected"  # target label

# Split ratios
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
SEED = 42

# Output paths
OUTPUT_DIR = "/content/faw_output"
SPLIT_DIR = f"{OUTPUT_DIR}/dataset_split"
MODEL_DIR = f"{OUTPUT_DIR}/models"

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
assert os.path.isdir(DRIVE_DATASET_PATH), (
    f"Dataset not found at: {DRIVE_DATASET_PATH}\n"
    f"Upload your dataset/ folder to Google Drive and update DRIVE_DATASET_PATH."
)

# Show source classes
classes = sorted(d for d in os.listdir(DRIVE_DATASET_PATH)
                 if os.path.isdir(os.path.join(DRIVE_DATASET_PATH, d)))
print(f"Found classes: {classes}")
for c in classes:
    count = len(os.listdir(os.path.join(DRIVE_DATASET_PATH, c)))
    print(f"  {c}: {count} images")

# Copy dataset from Drive to local SSD for faster I/O
LOCAL_DATASET = "/content/dataset_local"
if os.path.exists(LOCAL_DATASET):
    import shutil
    shutil.rmtree(LOCAL_DATASET)

print(f"\nCopying dataset from Drive to local disk (this is faster than reading from Drive during split)...")
t0 = time.time()
import shutil
shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET)
print(f"Done in {time.time() - t0:.1f}s")

print("\nNote: GPU will be used during training (steps 7-8). Data preparation steps use CPU only — this is normal.")

## 4. Split Dataset (Binary) + Oversample Minority

In [ ]:
import random
import shutil
from pathlib import Path
from collections import defaultdict

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def collect_images(class_dir):
    return sorted(p for p in Path(class_dir).rglob("*")
                  if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

# Use local copy (fast SSD) instead of Drive (slow network)
source_dir = Path(LOCAL_DATASET)
images_by_label = defaultdict(list)

for cls_dir in sorted(source_dir.iterdir()):
    if not cls_dir.is_dir():
        continue
    label = POSITIVE_LABEL if cls_dir.name.lower() == POSITIVE_CLASS.lower() else NEGATIVE_LABEL
    images_by_label[label].extend(collect_images(cls_dir))

# Split each class
split_dir = Path(SPLIT_DIR)
if split_dir.exists():
    shutil.rmtree(split_dir)

split_counts = {}
rng = random.Random(SEED)
t0 = time.time()

for label, images in sorted(images_by_label.items()):
    shuffled = list(images)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    splits = {
        "train": shuffled[:n_train],
        "val": shuffled[n_train:n_train + n_val],
        "test": shuffled[n_train + n_val:],
    }

    for split_name, split_imgs in splits.items():
        dest = split_dir / split_name / label
        dest.mkdir(parents=True, exist_ok=True)
        names_used = set()
        for img in split_imgs:
            name = img.name
            c = 1
            while name in names_used:
                name = f"{img.stem}_{c}{img.suffix}"
                c += 1
            names_used.add(name)
            shutil.copy2(img, dest / name)

    split_counts[label] = {s: len(imgs) for s, imgs in splits.items()}

# Oversample minority in train split
train_counts = {l: c["train"] for l, c in split_counts.items()}
max_count = max(train_counts.values())
min_label = min(train_counts, key=train_counts.get)

if train_counts[min_label] < max_count:
    minority_dir = split_dir / "train" / min_label
    existing = sorted(p for p in minority_dir.iterdir() if p.is_file())
    added = 0
    counter = 0
    names_used = {p.name for p in existing}
    while len(existing) + added < max_count:
        src = rng.choice(existing)
        counter += 1
        new_name = f"{src.stem}_dup{counter}{src.suffix}"
        while new_name in names_used:
            counter += 1
            new_name = f"{src.stem}_dup{counter}{src.suffix}"
        shutil.copy2(src, minority_dir / new_name)
        names_used.add(new_name)
        added += 1
    split_counts[min_label]["train_oversampled"] = train_counts[min_label] + added

print(f"Split + oversample done in {time.time() - t0:.1f}s")

# Print summary
print("\n" + "=" * 60)
print("  Dataset Split Summary")
print("=" * 60)
for label, counts in sorted(split_counts.items()):
    train_str = str(counts["train"])
    if "train_oversampled" in counts:
        train_str = f"{counts['train']} -> {counts['train_oversampled']}"
    print(f"  {label:<20} train={train_str:<15} val={counts['val']:<8} test={counts['test']}")
print("=" * 60)

## 5. Build tf.data Pipeline

In [ ]:
import tensorflow as tf
import numpy as np

print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_DIR}/train",
    labels="inferred", label_mode="int",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_DIR}/val",
    labels="inferred", label_mode="int",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE, shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_DIR}/test",
    labels="inferred", label_mode="int",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE, shuffle=False,
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print(f"\nClasses: {CLASS_NAMES}")
print(f"Num classes: {NUM_CLASSES}")

# Data augmentation (strong, to combat overfitting on small dataset)
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.25),
    tf.keras.layers.RandomContrast(0.25),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomTranslation(0.15, 0.15),
])

def augment_batch(images, labels):
    return augmentation(images, training=True), labels

aug_train_ds = (train_ds
    .map(augment_batch, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE))

val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

# Compute class weights from training labels
from sklearn.utils.class_weight import compute_class_weight

train_labels = []
for _, labels in train_ds.unbatch():
    train_labels.append(int(labels.numpy()))
train_labels = np.array(train_labels)

classes = np.unique(train_labels)
weights = compute_class_weight("balanced", classes=classes, y=train_labels)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print(f"\nClass weights: {class_weight}")
print(f"Train label distribution: {dict(zip(*np.unique(train_labels, return_counts=True)))}")

## 6. Define Model + Focal Loss

In [ ]:
@tf.keras.utils.register_keras_serializable(package="FAW")
class SparseFocalCrossEntropy(tf.keras.losses.Loss):
    """Focal loss for imbalanced classification.

    Down-weights well-classified (easy) examples so the model focuses
    on hard, typically minority-class, samples.
    """

    def __init__(self, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        true_probs = tf.gather(y_pred, y_true, batch_dims=1)
        focal_weight = tf.pow(1.0 - true_probs, self.gamma)
        loss = -focal_weight * tf.math.log(true_probs)
        return tf.reduce_mean(loss)

    def get_config(self):
        config = super().get_config()
        config["gamma"] = self.gamma
        return config


# Build MobileNetV2 model
base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
# Preprocessing: raw [0,255] pixels -> [-1,1] for MobileNetV2
# MobileNetV2 preprocess_input does: x / 127.5 - 1.0
# Rescaling is fully serializable (unlike Lambda layers)
x = tf.keras.layers.Rescaling(scale=1.0 / 127.5, offset=-1.0, name="preprocessing")(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(DROPOUT_RATE)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_FROZEN),
    loss=SparseFocalCrossEntropy(gamma=FOCAL_GAMMA),
    metrics=["accuracy"],
)

model.summary()

## 7. Phase 1: Train with Frozen Backbone

In [ ]:
import os
os.makedirs(MODEL_DIR, exist_ok=True)

callbacks_phase1 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=f"{MODEL_DIR}/best.keras",
        monitor="val_loss", mode="min",
        save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min",
        patience=5, restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=3, min_lr=1e-6, verbose=1,
    ),
]

print("Phase 1: Training with frozen backbone...")
history1 = model.fit(
    aug_train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    class_weight=class_weight,
    callbacks=callbacks_phase1,
)
print("Phase 1 complete.")

## 8. Phase 2: Fine-tune Top Layers

In [ ]:
# Unfreeze the top N layers of MobileNetV2
base_model.trainable = True
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Unfroze {trainable_count} of {len(base_model.layers)} base model layers")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_FINETUNE),
    loss=SparseFocalCrossEntropy(gamma=FOCAL_GAMMA),
    metrics=["accuracy"],
)

callbacks_phase2 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=f"{MODEL_DIR}/best.keras",
        monitor="val_loss", mode="min",
        save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min",
        patience=5, restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=3, min_lr=1e-7, verbose=1,
    ),
]

print("Phase 2: Fine-tuning top layers...")
history2 = model.fit(
    aug_train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    class_weight=class_weight,
    callbacks=callbacks_phase2,
)
print("Phase 2 complete.")

## 9. Evaluate on Test Set

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Load best model
best_model = tf.keras.models.load_model(
    f"{MODEL_DIR}/best.keras",
    custom_objects={"SparseFocalCrossEntropy": SparseFocalCrossEntropy},
)

# Predict
y_true, y_probs = [], []
for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_probs.extend(probs.tolist())

y_true = np.array(y_true)
y_probs = np.array(y_probs)
y_pred = np.argmax(y_probs, axis=1)

# Classification report
print("\n" + "=" * 60)
print("  Classification Report (argmax)")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Highlight infected recall
if POSITIVE_LABEL in CLASS_NAMES:
    infected_idx = CLASS_NAMES.index(POSITIVE_LABEL)
    infected_mask = y_true == infected_idx
    infected_recall = np.mean(y_pred[infected_mask] == infected_idx)
    print(f"\n>>> INFECTED RECALL: {infected_recall:.4f} ({int(infected_recall * infected_mask.sum())}/{infected_mask.sum()}) <<<")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/confusion_matrix.png", dpi=150)
plt.show()

# Threshold sweep for infected class
if POSITIVE_LABEL in CLASS_NAMES:
    infected_probs = y_probs[:, infected_idx]
    print("\nThreshold Sweep (infected class):")
    print(f"  {'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print("  " + "-" * 45)
    best_f1, best_thresh = 0, 0.5
    for thresh in np.arange(0.10, 0.95, 0.05):
        pred_infected = infected_probs >= thresh
        actual_infected = y_true == infected_idx
        tp = (pred_infected & actual_infected).sum()
        fp = (pred_infected & ~actual_infected).sum()
        fn = (~pred_infected & actual_infected).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        print(f"  {thresh:>10.2f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    print(f"\n  >>> Recommended threshold: {best_thresh:.2f} (F1={best_f1:.4f}) <<<")

# ROC curve
if POSITIVE_LABEL in CLASS_NAMES:
    fpr, tpr, _ = roc_curve(y_true == infected_idx, infected_probs)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve — Infected Detection")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{MODEL_DIR}/roc_curve.png", dpi=150)
    plt.show()

## 10. Training Curves

In [ ]:
# Merge histories from both phases
history = {}
for key in history1.history:
    history[key] = history1.history[key] + history2.history.get(key, [])

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.get("loss", []), label="train_loss")
plt.plot(history.get("val_loss", []), label="val_loss")
plt.axvline(x=len(history1.history.get("loss", [])) - 0.5,
            color="red", linestyle="--", alpha=0.5, label="Fine-tune start")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.get("accuracy", []), label="train_acc")
plt.plot(history.get("val_accuracy", []), label="val_acc")
plt.axvline(x=len(history1.history.get("accuracy", [])) - 0.5,
            color="red", linestyle="--", alpha=0.5, label="Fine-tune start")
plt.title("Accuracy")
plt.legend()

plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/training_curves.png", dpi=150)
plt.show()

# Save history JSON
import json
with open(f"{MODEL_DIR}/history.json", "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.items()}, f, indent=2)

## 11. Export Model

Download the model files to use locally with the Streamlit app.

In [ ]:
import json

# Save labels
labels_path = f"{MODEL_DIR}/labels.json"
with open(labels_path, "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)
print(f"Labels saved: {labels_path}")
print(f"Model saved: {MODEL_DIR}/best.keras")

# Copy to Google Drive for persistence
drive_output = "/content/drive/MyDrive/FAW_dataset/trained_model"
os.makedirs(drive_output, exist_ok=True)
for fname in ["best.keras", "labels.json", "history.json",
              "training_curves.png", "confusion_matrix.png", "roc_curve.png"]:
    src = f"{MODEL_DIR}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, f"{drive_output}/{fname}")
print(f"\nCopied to Drive: {drive_output}")

# Download to local machine
print("\nDownloading model files...")
from google.colab import files
files.download(f"{MODEL_DIR}/best.keras")
files.download(f"{MODEL_DIR}/labels.json")

## Done!

**Next steps:**
1. Place `best.keras` and `labels.json` in `models/mobilenetv2_binary_run/` on your local machine
2. Run: `streamlit run app.py`
3. Upload an image and check the prediction
4. Adjust the **Infected detection threshold** slider if needed (recommended value shown above)